# Multi-seed YOLOv8 — Seed 2 (training notebook)

Part of a **3-notebook controlled experiment** (seed 1, 2, 3). Every setting is identical across the three notebooks; **only `SEED` changes** (different weight init + augmentation order).

**Before running:** set Runtime → Change runtime type → **GPU (T4)**.

**One-time data upload:** zip your local `data/yolo/` folder (the YOLO dataset built by `coco_to_yolo.py`, containing `images/{train,val,test}`, `labels/{train,val,test}`, `data.yaml`) and the test COCO GT `data/arcade/stenosis/test/annotations/test.json`, and put them in your Google Drive under the `PROJECT` path set below.


In [ ]:
# 1. Dependencies (pin Ultralytics to match the local pipeline)
!pip -q install ultralytics==8.1.34 ensemble-boxes==1.0.9 pycocotools==2.0.7


In [ ]:
# 2. Authenticate and mount Drive
import os

try:
    # Explicitly trigger OAuth popup — bypasses silent credential propagation failures
    from google.colab import auth
    auth.authenticate_user()
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/arcade_multiseed'   # <-- edit to taste
    print(f'Drive mounted. Project root: {PROJECT}')
except Exception as e:
    PROJECT = '/content/arcade_multiseed'
    print(f'Drive unavailable ({e.__class__.__name__}) — using local Colab storage: {PROJECT}')
    print('Weights/outputs will be lost when the session ends.')
    print()
    print('To get your data into Colab without Drive, run this in a new cell:')
    print('  from google.colab import files')
    print('  files.upload()   # select your arcade_multiseed.zip')
    print('  import zipfile')
    print('  zipfile.ZipFile("/content/arcade_multiseed.zip").extractall("/content/")')

os.makedirs(PROJECT, exist_ok=True)
DATA_YAML = f'{PROJECT}/data/yolo/data.yaml'
GT_COCO   = f'{PROJECT}/data/test.json'
OUT = f'{PROJECT}/outputs'; os.makedirs(OUT, exist_ok=True)

if not os.path.exists(DATA_YAML):
    raise RuntimeError(
        f'data.yaml not found at {DATA_YAML}\n'
        'Upload your data/yolo folder first — see instructions printed above.'
    )
print('data.yaml OK; outputs ->', OUT)

In [ ]:
# 3. PyTorch>=2.6 loads checkpoints with weights_only=True by default; patch for Ultralytics
import torch
_orig = torch.load
def _patched(*a, **k):
    k.setdefault('weights_only', False)
    return _orig(*a, **k)
torch.load = _patched


In [ ]:
# 4. THE ONLY KNOB THAT DIFFERS BETWEEN THE THREE NOTEBOOKS
SEED = 2
print('training with seed =', SEED)


## Train (identical settings; only `SEED` differs)


In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8m.pt')   # COCO-pretrained start (same for all seeds)
results = model.train(
    data=DATA_YAML,
    epochs=100, imgsz=640, batch=16,
    optimizer='SGD', lr0=0.01, lrf=0.01,
    patience=30, close_mosaic=10,
    seed=SEED, deterministic=True,
    project=f'{OUT}/runs', name=f'seed{SEED}', exist_ok=True,
)


In [ ]:
# Save the best checkpoint as best_seed{SEED}.pt on Drive
import shutil
best = f'{OUT}/runs/seed{SEED}/weights/best.pt'
dst  = f'{OUT}/best_seed{SEED}.pt'
shutil.copy(best, dst)
print('saved', dst)


## Evaluate on the TEST set → metrics_seed{SEED}.json


In [ ]:
import json
m = YOLO(dst).val(data=DATA_YAML, split='test', verbose=False)
P, R = float(m.box.mp), float(m.box.mr)
metrics = {'seed': SEED, 'precision': P, 'recall': R,
           'f1': 2*P*R/(P+R+1e-9),
           'map50': float(m.box.map50), 'map50_95': float(m.box.map)}
json.dump(metrics, open(f'{OUT}/metrics_seed{SEED}.json','w'), indent=2)
print(metrics)


## Dump per-image predictions → pred_seed{SEED}.json (COCO detection format)
Confidence is kept very low (0.001) so the full precision–recall curve is preserved for fair mAP and for later Weighted Box Fusion.


In [ ]:
import glob, yaml
d = yaml.safe_load(open(DATA_YAML)); root = d['path']
test_dir = os.path.join(root, d['test'])
imgs = sorted(glob.glob(os.path.join(test_dir, '*.png')) + glob.glob(os.path.join(test_dir, '*.jpg')))
mdl = YOLO(dst)
preds = []
for ip in imgs:
    stem = os.path.splitext(os.path.basename(ip))[0]
    r = mdl.predict(ip, imgsz=640, conf=0.001, verbose=False)[0]
    for b in r.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        preds.append({'image_id': stem, 'category_id': 0,
                      'bbox': [x1, y1, x2-x1, y2-y1], 'score': float(b.conf)})
json.dump(preds, open(f'{OUT}/pred_seed{SEED}.json','w'))
print(len(preds), 'detections ->', f'{OUT}/pred_seed{SEED}.json')


Done. Run the **other two notebooks** (seed 2, seed 3), then open `fusion_compare.ipynb` to compare seed variability and fuse with WBF.
